In [1]:
import os
import tqdm
import pickle
import numpy as np

In [6]:
exp_path = '../../boxes/others/'

In [ ]:
import glob
import pandas as pd
dfs = []

for csv in glob.glob('./real_user_data/*.csv'):
    
    df = pd.read_csv(csv)

    df_no_tight = df[df['image_path'] != df['masks_path'].apply(lambda x: x.replace('masks', 'images'))].copy()

    df_no_tight['dataset'] = df_no_tight['image_path'].apply(lambda x: x.split('images/')[-1].split('_')[0])

    print('------------------------------')
    csv_ = csv.split("batched_user_output_")
    csv_ = "_".join(csv_)
    print(csv_.split(".")[0])
    
    for dataset in [
                    'GrabCut',
                    'Berkeley',
                    'DAVIS',
                    'COCO',
                    'TETRIS',
                    'PASCAL',
                    'ADE20K',

                    'ACDC',
                    'BUID',
                    'MedScribble'
                   ]:

        df_dataset = df_no_tight.loc[df_no_tight['dataset'] == dataset]

        df_dataset = df_dataset.groupby('image_path')['iou']
        
        print(dataset, "%.2f" % (100 * df_dataset.mean().mean()) + '±' + "%.2f" % (100 * df_dataset.std().mean()))

In [7]:
data_dict = {}

for model_name in tqdm.tqdm(set([x.replace("_MAX", "").replace("_MIN", "") for x in os.listdir(exp_path)])):
    print(model_name)
    data_dict[model_name] = {}

    for suffix in ['MAX', 'MIN']:

        model_path = os.path.join(exp_path, model_name + '_' + suffix, 'plots')

        for dataset_name in os.listdir(model_path):

            base_trajectory_data = None

            if dataset_name.split('_')[0] not in data_dict[model_name]:
                data_dict[model_name][dataset_name.split('_')[0]] = {}

            list_per_click = []
            list_per_click_base = []

            list_per_click_biou = []
            list_per_click_base_biou = []
    
            try:
                data = pickle.load(open(os.path.join(model_path, dataset_name), "rb"))
            except:
                continue
            for per_image_metrics in data['all_ious']:                        
                list_per_click.append(per_image_metrics[1][:, 0])
                list_per_click_biou.append(per_image_metrics[1][:, 1])

            
            for per_image_metrics in data['all_ious']:
                list_per_click_base.append(per_image_metrics[1][:, 2][0][:, 0][0])                
                list_per_click_base_biou.append(per_image_metrics[1][:, 2][0][:, 1][0])

            data_dict[model_name][dataset_name.split('_')[0]][suffix + '_IOU'] = np.array(list_per_click)
            data_dict[model_name][dataset_name.split('_')[0]][suffix + '_BIOU'] = np.array(list_per_click_biou)

            # if base_trajectory_data is not None:
            
            data_dict[model_name][dataset_name.split('_')[0]]['STANDARD_IOU'] = np.array(list_per_click_base)
            data_dict[model_name][dataset_name.split('_')[0]]['STANDARD_BIOU'] = np.array(list_per_click_base_biou)

  0%|                                                                                                                                                                | 0/15 [00:00<?, ?it/s]

robustsam_checkpoint_h


  7%|██████████▏                                                                                                                                             | 1/15 [00:01<00:14,  1.07s/it]

mobile_sam


 13%|████████████████████▎                                                                                                                                   | 2/15 [00:02<00:13,  1.03s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                    | 13/15 [00:15<00:02,  1.20s/it]

sam_vit_b_01ec64


 93%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉          | 14/15 [00:16<00:01,  1.18s/it]

sam_vit_h_4b8939


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:17<00:00,  1.19s/it]


In [13]:
inverse_index = {}

for idx, model_name in enumerate(data_dict.keys()):
        
    for dataset in data_dict[model_name].keys():
        
        if dataset not in inverse_index:
            inverse_index[dataset] = {}
            
        if model_name not in inverse_index[dataset]:
            inverse_index[dataset][model_name] = data_dict[model_name][dataset]

In [14]:
all_datasets = list(data_dict[model_name].keys())

In [20]:
models_to_print = set([x.replace("_MAX", "").replace("_MIN", "") for x in os.listdir(exp_path)])
datasets_to_print = ["TETRIS", "GrabCut", "DAVIS"] 


In [21]:
models_to_print= list(models_to_print)
models_to_print.sort()
models_to_print

['mobile_sam',
 'robustsam_checkpoint_b',
 'robustsam_checkpoint_h',
 'robustsam_checkpoint_l',
 'sam2.1_hiera_base_plus',
 'sam2.1_hiera_large',
 'sam2.1_hiera_small',
 'sam2.1_hiera_tiny',
 'sam2.1_hq_hiera_large',
 'sam_hq_vit_b',
 'sam_hq_vit_h',
 'sam_hq_vit_l',
 'sam_vit_b_01ec64',
 'sam_vit_h_4b8939',
 'sam_vit_l_0b3195']

In [27]:
sep = '-' * 32
for idx, dataset in enumerate(datasets_to_print):
    
    print(sep)
    print(dataset)
    
    for model_name in models_to_print:
        metrics_vals = {}
        print(sep)
        print(model_name)
        
        for metric in ['IOU']:

            if not ('MAX_' + metric in inverse_index[dataset][model_name] and 'MIN_' + metric in inverse_index[dataset][model_name]):
                continue

            max_area = inverse_index[dataset][model_name]['MAX_' + metric].mean(axis=0).shape[0] - 1
            # print(inverse_index[dataset][model_name]['MAX_' + metric])
            MAX = inverse_index[dataset][model_name]['MAX_' + metric].mean(axis=0)[0]
            MIN = inverse_index[dataset][model_name]['MIN_' + metric].mean(axis=0)[0]
            

            BASE = inverse_index[dataset][model_name]['STANDARD_' + metric].mean(axis=0)
            DELTA = MAX - MIN
            
            print(metric.ljust(4), '| Min', '{:.2f}'.format(100 * (MIN)), 
                  '| Base', '{:.2f}'.format(100 * (BASE)),
                  '| Max', '{:.2f}'.format(100 * (MAX)),
                  '| Delta', '{:.2f}'.format(100 * (MAX - MIN)))

--------------------------------
TETRIS
--------------------------------
mobile_sam
[np.float64(0.9315955815627249)]
[np.float64(0.809803860230863)]
IOU  | Min 61.15 | Base 91.01 | Max 92.90 | Delta 31.75
--------------------------------
robustsam_checkpoint_b
[np.float64(0.8057309373482273)]
[np.float64(0.11983744163691294)]
IOU  | Min 26.44 | Base 80.26 | Max 84.33 | Delta 57.89
--------------------------------
robustsam_checkpoint_h
[np.float64(0.18448638531387082)]
[np.float64(0.1345866918014633)]
IOU  | Min 25.30 | Base 46.86 | Max 51.71 | Delta 26.42
--------------------------------
robustsam_checkpoint_l
[np.float64(0.5782722676487111)]
[np.float64(0.05823658613115783)]
IOU  | Min 3.88 | Base 62.81 | Max 68.44 | Delta 64.56
--------------------------------
sam2.1_hiera_base_plus
[np.float64(0.9608733788609736)]
[np.float64(0.0061732074846828945)]
IOU  | Min 37.55 | Base 92.72 | Max 94.63 | Delta 57.08
--------------------------------
sam2.1_hiera_large
[np.float64(0.953157268915

In [30]:
import numpy as np

sep = '-' * 32
per_model = {m: {'min': [], 'tight': [], 'max': [], 'delta': []}
             for m in models_to_print}

for idx, dataset in enumerate(datasets_to_print):
    if dataset not in inverse_index:
        continue

    for model_name in models_to_print:
        if model_name not in inverse_index[dataset]:
            continue

        for metric in ['IOU']:
            if not (
                'MAX_' + metric in inverse_index[dataset][model_name]
                and 'MIN_' + metric in inverse_index[dataset][model_name]
                and 'STANDARD_' + metric in inverse_index[dataset][model_name]
            ):
                continue
            
            max_vals = inverse_index[dataset][model_name]['MAX_' + metric].mean(axis=0)
            min_vals = inverse_index[dataset][model_name]['MIN_' + metric].mean(axis=0)
            base_vals = inverse_index[dataset][model_name]['STANDARD_' + metric].mean(axis=0)

            MAX = max_vals[0]
            MIN = min_vals[0]
            TIGHT = base_vals        
            DELTA = MAX - MIN

            per_model[model_name]['min'].append(MIN)
            per_model[model_name]['tight'].append(TIGHT)
            per_model[model_name]['max'].append(MAX)
            per_model[model_name]['delta'].append(DELTA)

avg_metrics = {}
for model_name in models_to_print:
    vals = per_model[model_name]
    if not vals['min']:
    
        continue

    
    min_mean = float(np.mean(vals['min']))
    max_mean = float(np.mean(vals['max']))
    delta_mean = float(np.mean(vals['delta']))

    
    tight_mean = np.mean(np.stack(vals['tight'], axis=0), axis=0)

    avg_metrics[model_name] = {
        'IOU_min': min_mean,
        'IOU_tight': tight_mean,
        'IOU_max': max_mean,
        'IOU_delta': delta_mean,
    }

    
    print(sep)
    print(model_name)
    print(
        'IOU'.ljust(4),
        '| Min',   '{:.2f}'.format(100 * min_mean),
        '| Tight', '{:.2f}'.format(100 * np.mean(tight_mean)),
        '| Max',   '{:.2f}'.format(100 * max_mean),
        '| Delta', '{:.2f}'.format(100 * delta_mean),
    )




--------------------------------
sam2.1_hiera_tiny
IOU  | Min 47.49 | Tight 83.30 | Max 86.37 | Delta 38.88
--------------------------------
sam_vit_l_0b3195
IOU  | Min 61.74 | Tight 80.89 | Max 84.99 | Delta 23.25
--------------------------------
robustsam_checkpoint_h
IOU  | Min 25.09 | Tight 37.96 | Max 42.01 | Delta 16.91
--------------------------------
sam_hq_vit_l
IOU  | Min 64.59 | Tight 83.18 | Max 86.44 | Delta 21.85
--------------------------------
sam_vit_b_01ec64
IOU  | Min 54.38 | Tight 77.46 | Max 82.84 | Delta 28.46
--------------------------------
sam_hq_vit_b
IOU  | Min 58.46 | Tight 75.84 | Max 82.34 | Delta 23.87
--------------------------------
sam2.1_hiera_base_plus
IOU  | Min 48.50 | Tight 82.38 | Max 86.29 | Delta 37.79
--------------------------------
robustsam_checkpoint_l
IOU  | Min 5.81 | Tight 55.71 | Max 61.27 | Delta 55.46
--------------------------------
sam2.1_hq_hiera_large
IOU  | Min 53.73 | Tight 85.10 | Max 88.18 | Delta 34.46
----------------------